# MGFR Classifier Sanity Check
### He et al. 2017 CelebA ResNet-50 — Full Evaluation on Kaggle
Loads the CelebA dataset via `kagglehub`, runs the converted classifier,
and reports per-attribute accuracy against ground-truth labels.

In [1]:
# ── Install dependencies ────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'kagglehub', 'torch', 'torchvision', 'numpy',
                'matplotlib', 'Pillow', 'scikit-learn'], check=True)

import os, json
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import kagglehub

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

Device: cpu
PyTorch: 2.9.0+cpu


## Step 1 — Upload your converted weights
Run the cell below, then use the file picker to upload `celeba28_resnet50.pth`
(the file produced by `convert_caffe.py` on your local machine).

In [2]:
from google.colab import files   # works on Colab; on Kaggle just upload via sidebar
import os

# ── Option A: Kaggle — add the .pth as a dataset input via the sidebar,
#             then set the path below manually:
WEIGHTS_PATH = '/kaggle/input/your-dataset/celeba28_resnet50.pth'

# ── Option B: Colab — uncomment the upload widget:
# uploaded = files.upload()
# WEIGHTS_PATH = list(uploaded.keys())[0]

# ── Option C: If running locally, just set the path:
# WEIGHTS_PATH = 'celeba28_resnet50.pth'

assert os.path.exists(WEIGHTS_PATH), f'File not found: {WEIGHTS_PATH}'
print(f'Weights file found: {os.path.getsize(WEIGHTS_PATH)/1e6:.1f} MB')

AssertionError: File not found: /kaggle/input/your-dataset/celeba28_resnet50.pth

## Step 2 — Download CelebA dataset via KaggleHub

In [ ]:
path = kagglehub.dataset_download('jessicali9530/celeba-dataset')
print(f'Dataset downloaded to: {path}')

# Discover directory structure
for root, dirs, files in os.walk(path):
    dirs[:] = [d for d in dirs if d != '__pycache__']
    level = root.replace(path, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in files[:5]:
            print(f'{indent}  {f}')
        if len(files) > 5:
            print(f'{indent}  ... ({len(files)} files total)')

## Step 3 — Locate images and label file

In [ ]:
# Find img_align_celeba directory
IMG_DIR = None
for root, dirs, files in os.walk(path):
    if 'img_align_celeba' in dirs:
        IMG_DIR = os.path.join(root, 'img_align_celeba')
        break
    if os.path.basename(root) == 'img_align_celeba' and files:
        IMG_DIR = root
        break

# Find list_attr_celeba.txt (ground-truth labels)
LABEL_FILE = None
for root, dirs, files in os.walk(path):
    for f in files:
        if 'attr' in f.lower() and f.endswith('.txt') or f.endswith('.csv'):
            LABEL_FILE = os.path.join(root, f)
            break

print(f'Image directory : {IMG_DIR}')
print(f'Label file      : {LABEL_FILE}')

assert IMG_DIR   is not None, 'Could not find img_align_celeba — check dataset structure'
assert LABEL_FILE is not None, 'Could not find attribute label file'

img_files = sorted(os.listdir(IMG_DIR))
print(f'Images found    : {len(img_files)}')
print(f'First few       : {img_files[:5]}')

## Step 4 — Attribute definitions and CelebA-40 → Paper-28 mapping

In [ ]:
CELEBA_40 = [
    '5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive',    'Bags_Under_Eyes',
    'Bald',             'Bangs',           'Big_Lips',      'Big_Nose',
    'Black_Hair',       'Blond_Hair',      'Blurry',        'Brown_Hair',
    'Bushy_Eyebrows',   'Chubby',          'Double_Chin',   'Eyeglasses',
    'Goatee',           'Gray_Hair',       'Heavy_Makeup',  'High_Cheekbones',
    'Male',             'Mouth_Slightly_Open', 'Mustache',  'Narrow_Eyes',
    'No_Beard',         'Oval_Face',       'Pale_Skin',     'Pointy_Nose',
    'Receding_Hairline','Rosy_Cheeks',     'Sideburns',     'Smiling',
    'Straight_Hair',    'Wavy_Hair',       'Wearing_Earrings', 'Wearing_Hat',
    'Wearing_Lipstick', 'Wearing_Necklace','Wearing_Necktie',  'Young',
]

PAPER_28_TO_CELEBA40 = {
    'Black Hair':          'Black_Hair',
    'Blond Hair':          'Blond_Hair',
    'Blurry':              'Blurry',
    'Brown Hair':          'Brown_Hair',
    'Eyeglasses':          'Eyeglasses',
    'Gray Hair':           'Gray_Hair',
    'Heavy Makeup':        'Heavy_Makeup',
    'Mouth Slightly Open': 'Mouth_Slightly_Open',
    'Mustache':            'Mustache',
    'Big Eyes':            'Narrow_Eyes',   # Big Eyes = NOT Narrow_Eyes
    'No Beard':            'No_Beard',
    'Receding Hairline':   'Receding_Hairline',
    'Sideburns':           'Sideburns',
    'Smiling':             'Smiling',
    'Straight Hair':       'Straight_Hair',
    'Wearing Earrings':    'Wearing_Earrings',
    'Wearing Hat':         'Wearing_Hat',
    'Male':                'Male',
    'Wearing Necklace':    'Wearing_Necklace',
    'Big Nose':            'Big_Nose',
    'Wearing Lipstick':    'Wearing_Lipstick',
    'Young':               'Young',
    'Wavy Hair':           'Wavy_Hair',
    'Big Lips':            'Big_Lips',
    'Bald':                'Bald',
    'Bangs':               'Bangs',
    'Chubby':              'Chubby',
    'Double Chin':         'Double_Chin',
}

PAPER_28        = list(PAPER_28_TO_CELEBA40.keys())
CELEBA40_IDX    = {name: i for i, name in enumerate(CELEBA_40)}
ATTR_28_INDICES = [CELEBA40_IDX[PAPER_28_TO_CELEBA40[a]] for a in PAPER_28]
INVERTED_IDX    = [i for i, a in enumerate(PAPER_28) if a == 'Big Eyes']

print(f'28 attributes defined')
print(f'CelebA-40 indices: {ATTR_28_INDICES}')

## Step 5 — Parse label file

In [ ]:
def parse_label_file(label_path):
    """
    Handles both formats:
      - Standard CelebA: first line = num_images, second = attr names, rest = data
      - Raw format: filename + 40 values per line (no header)
    """
    filenames, rows = [], []
    with open(label_path, 'r') as f:
        lines = [l.strip() for l in f if l.strip()]

    # Detect format: if first line is a pure integer it's standard CelebA format
    start = 0
    if lines[0].strip().isdigit():
        start = 2   # skip count line + header line

    for line in lines[start:]:
        parts = line.split()
        if len(parts) < 41:
            continue
        filenames.append(parts[0])
        vals = [1 if int(x) == 1 else 0 for x in parts[1:41]]
        rows.append(vals)

    labels_40 = np.array(rows, dtype=np.int32)
    print(f'Parsed {labels_40.shape[0]} images x {labels_40.shape[1]} attributes')
    return filenames, labels_40


def extract_28(labels_40):
    labels_28 = labels_40[:, ATTR_28_INDICES].copy()
    for idx in INVERTED_IDX:
        labels_28[:, idx] = 1 - labels_28[:, idx]
    return labels_28


all_filenames, labels_40 = parse_label_file(LABEL_FILE)
labels_28 = extract_28(labels_40)

print(f'\nPer-attribute positive rates (should look like real CelebA stats):')
for i, attr in enumerate(PAPER_28):
    rate = labels_28[:, i].mean()
    bar  = '█' * int(rate * 30)
    print(f'  {attr:<25} {rate*100:5.1f}%  {bar}')

## Step 6 — Get test split (last 19,962 images = official CelebA test set)

In [ ]:
# Official CelebA split: train=162770, val=19867, test=19962
# We evaluate on the test split only
N_TRAIN = 162770
N_VAL   = 19867
N_TEST  = 19962

test_filenames = all_filenames[N_TRAIN + N_VAL : N_TRAIN + N_VAL + N_TEST]
test_labels    = labels_28[N_TRAIN + N_VAL : N_TRAIN + N_VAL + N_TEST]

# Build fast lookup: filename -> index in test set
test_file_set = set(test_filenames)

print(f'Test set size : {len(test_filenames)}')
print(f'Label shape   : {test_labels.shape}')
print(f'Sample files  : {test_filenames[:3]}')

## Step 7 — Load the converted classifier

In [ ]:
def load_classifier(weights_path, device):
    state    = torch.load(weights_path, map_location=device)
    fc_out   = state['fc.weight'].shape[0]
    fc_in    = state['fc.weight'].shape[1]
    print(f'FC layer shape: ({fc_out}, {fc_in})')

    model = models.resnet50(weights=None)
    model.fc = nn.Linear(2048, fc_out)
    missing, unexpected = model.load_state_dict(state, strict=False)

    if missing:
        print(f'Missing keys  : {len(missing)} (usually just BN buffers — OK)')
    if unexpected:
        print(f'Unexpected    : {unexpected}')

    model.to(device).eval()
    print(f'Model loaded on {device}')
    return model, fc_out


TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

model, fc_out = load_classifier(WEIGHTS_PATH, DEVICE)

## Step 8 — Quick forward pass sanity check (no images needed)

In [ ]:
with torch.no_grad():
    dummy = torch.randn(4, 3, 224, 224).to(DEVICE)
    out   = torch.sigmoid(model(dummy)).cpu().numpy()

print(f'Output shape  : {out.shape}  (expected [4, {fc_out}])')
print(f'Output range  : [{out.min():.4f}, {out.max():.4f}]  (expected [0, 1])')
print(f'Mean prob     : {out.mean():.4f}')
assert out.shape[1] == fc_out, 'Shape mismatch!'
assert out.min() >= 0 and out.max() <= 1, 'Probabilities out of range!'
print('\n✓ Forward pass OK')

## Step 9 — Full evaluation on CelebA test set

In [ ]:
BATCH_SIZE  = 64
THRESHOLD   = 0.5
MAX_IMAGES  = None   # set to e.g. 500 for a quick check, None for full test set

eval_files  = test_filenames[:MAX_IMAGES] if MAX_IMAGES else test_filenames
eval_labels = test_labels[:MAX_IMAGES]    if MAX_IMAGES else test_labels

all_probs = []
skipped   = 0

for start in range(0, len(eval_files), BATCH_SIZE):
    batch_fnames = eval_files[start:start + BATCH_SIZE]
    tensors = []
    for fname in batch_fnames:
        img_path = os.path.join(IMG_DIR, fname)
        try:
            img = Image.open(img_path).convert('RGB')
            tensors.append(TRANSFORM(img))
        except Exception as e:
            tensors.append(torch.zeros(3, 224, 224))
            skipped += 1

    batch = torch.stack(tensors).to(DEVICE)
    with torch.no_grad():
        probs = torch.sigmoid(model(batch)).cpu().numpy()

    # If model outputs 40, slice to 28
    if probs.shape[1] == 40:
        probs_28 = probs[:, ATTR_28_INDICES]
        for idx in INVERTED_IDX:
            probs_28[:, idx] = 1.0 - probs_28[:, idx]
        probs = probs_28

    all_probs.append(probs)

    done = min(start + BATCH_SIZE, len(eval_files))
    if done % 1000 == 0 or done == len(eval_files):
        print(f'  {done}/{len(eval_files)}', end='\r')

all_probs = np.vstack(all_probs)
preds     = (all_probs >= THRESHOLD).astype(int)
print(f'\nDone. Skipped {skipped} images.')
print(f'Predictions shape: {preds.shape}')

## Step 10 — Compute per-attribute metrics

In [ ]:
results = {}
for i, attr in enumerate(PAPER_28):
    gt_i   = eval_labels[:, i]
    pred_i = preds[:, i]

    tp = int(((pred_i == 1) & (gt_i == 1)).sum())
    fp = int(((pred_i == 1) & (gt_i == 0)).sum())
    fn = int(((pred_i == 0) & (gt_i == 1)).sum())
    tn = int(((pred_i == 0) & (gt_i == 0)).sum())

    acc  = (tp + tn) / (tp + fp + fn + tn + 1e-9)
    prec = tp / (tp + fp + 1e-9)
    rec  = tp / (tp + fn + 1e-9)
    f1   = 2 * prec * rec / (prec + rec + 1e-9)

    results[attr] = dict(
        accuracy=round(acc, 4), precision=round(prec, 4),
        recall=round(rec, 4),   f1=round(f1, 4),
        tp=tp, fp=fp, fn=fn, tn=tn,
        gt_pos_rate=round(float(gt_i.mean()), 4),
        pred_pos_rate=round(float(pred_i.mean()), 4),
    )

mean_acc  = np.mean([v['accuracy']  for v in results.values()])
mean_f1   = np.mean([v['f1']        for v in results.values()])
mean_prec = np.mean([v['precision'] for v in results.values()])
mean_rec  = np.mean([v['recall']    for v in results.values()])

print('='*70)
print(f'  OVERALL   Acc={mean_acc:.4f}  F1={mean_f1:.4f}  '
      f'Prec={mean_prec:.4f}  Rec={mean_rec:.4f}')
print('='*70)
print(f'  {"ATTRIBUTE":<25} {"ACC":>6} {"F1":>6} {"PREC":>6} {"REC":>6}')
print('-'*70)
for attr in sorted(results, key=lambda a: results[a]['accuracy']):
    m    = results[attr]
    flag = '  <-- CHECK' if m['accuracy'] < 0.70 else ''
    print(f'  {attr:<25} {m["accuracy"]:>6.4f} {m["f1"]:>6.4f} '
          f'{m["precision"]:>6.4f} {m["recall"]:>6.4f}{flag}')
print('='*70)

if mean_acc > 0.80:
    print('\n✓ Accuracy > 80% — conversion successful')
elif mean_acc > 0.70:
    print('\n⚠ Accuracy 70-80% — try re-running with fix_bgr=True below')
else:
    print('\n✗ Accuracy < 70% — FC heads likely not transferred, check layer names')

## Step 11 — Visualise per-attribute accuracy

In [ ]:
attrs_sorted = sorted(results, key=lambda a: results[a]['accuracy'], reverse=True)
accs  = [results[a]['accuracy']  for a in attrs_sorted]
f1s   = [results[a]['f1']        for a in attrs_sorted]

x = np.arange(len(attrs_sorted))
width = 0.4

fig, ax = plt.subplots(figsize=(16, 6))
bars_acc = ax.bar(x - width/2, accs, width, label='Accuracy', color='steelblue', alpha=0.85)
bars_f1  = ax.bar(x + width/2, f1s,  width, label='F1',       color='coral',     alpha=0.85)

ax.axhline(0.80, color='green',  linestyle='--', linewidth=1.2, label='80% threshold')
ax.axhline(mean_acc, color='steelblue', linestyle=':', linewidth=1.2,
           label=f'Mean acc = {mean_acc:.3f}')

ax.set_xticks(x)
ax.set_xticklabels(attrs_sorted, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.set_title('Per-Attribute Accuracy & F1 — CelebA Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('attribute_accuracy.png', dpi=150)
plt.show()
print('Saved: attribute_accuracy.png')

## Step 12 — Prediction vs ground-truth rate heatmap

In [ ]:
gt_rates   = [results[a]['gt_pos_rate']   for a in PAPER_28]
pred_rates = [results[a]['pred_pos_rate']  for a in PAPER_28]

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(PAPER_28))
ax.bar(x - 0.2, gt_rates,   0.4, label='Ground truth',  color='steelblue', alpha=0.85)
ax.bar(x + 0.2, pred_rates, 0.4, label='Predicted',     color='coral',     alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(PAPER_28, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Positive rate')
ax.set_title('Ground Truth vs Predicted Positive Rate per Attribute')
ax.legend()
plt.tight_layout()
plt.savefig('positive_rates.png', dpi=150)
plt.show()

## Step 13 — Visual check: predictions on sample images

In [ ]:
N_SAMPLES = 6
sample_files  = test_filenames[:N_SAMPLES]
sample_labels = test_labels[:N_SAMPLES]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (fname, gt) in enumerate(zip(sample_files, sample_labels)):
    img_path = os.path.join(IMG_DIR, fname)
    img = Image.open(img_path).convert('RGB')

    tensor = TRANSFORM(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob = torch.sigmoid(model(tensor)).cpu().numpy()[0]

    if len(prob) == 40:
        prob_28 = prob[ATTR_28_INDICES]
        for i in INVERTED_IDX:
            prob_28[i] = 1.0 - prob_28[i]
        prob = prob_28

    pred = (prob >= 0.5).astype(int)

    # Build annotation: show predicted positives, mark wrong ones
    lines = []
    for i, attr in enumerate(PAPER_28):
        if pred[i] == 1 or gt[i] == 1:
            p_str = f'{prob[i]:.2f}'
            match = '✓' if pred[i] == gt[i] else '✗'
            lines.append(f'{match} {attr}: {p_str}')

    ax = axes[idx]
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(fname, fontsize=8)

    # Overlay text
    text = '\n'.join(lines[:12])   # limit to 12 lines
    ax.text(1.02, 0.98, text, transform=ax.transAxes,
            fontsize=7, verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('Sample Predictions (✓=correct, ✗=wrong)', fontsize=12)
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 14 — Optional: re-run with Caffe BGR normalisation
If overall accuracy is 70–80%, the issue may be that the Caffe model
was trained with BGR mean subtraction instead of PyTorch's RGB normalisation.
Run this cell to test the alternative normalisation.

In [ ]:
TRANSFORM_BGR = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[123.68/255, 116.779/255, 103.939/255],  # Caffe ImageNet BGR->RGB
        std =[1.0, 1.0, 1.0]
    ),
])

# Quick check on 200 images
quick_probs, quick_gt = [], []
for fname, gt in zip(test_filenames[:200], test_labels[:200]):
    img_path = os.path.join(IMG_DIR, fname)
    try:
        img = Image.open(img_path).convert('RGB')
        t   = TRANSFORM_BGR(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            p = torch.sigmoid(model(t)).cpu().numpy()[0]
        if len(p) == 40:
            p28 = p[ATTR_28_INDICES]
            for i in INVERTED_IDX: p28[i] = 1.0 - p28[i]
            p = p28
        quick_probs.append(p)
        quick_gt.append(gt)
    except: pass

quick_probs = np.array(quick_probs)
quick_gt    = np.array(quick_gt)
quick_preds = (quick_probs >= 0.5).astype(int)
bgr_acc     = (quick_preds == quick_gt).mean()

# Compare to original normalisation
orig_preds = (all_probs[:200] >= 0.5).astype(int)
orig_acc   = (orig_preds == test_labels[:200]).mean()

print(f'Accuracy with standard RGB normalisation : {orig_acc:.4f}')
print(f'Accuracy with Caffe BGR  normalisation   : {bgr_acc:.4f}')

if bgr_acc > orig_acc + 0.02:
    print('\n→ BGR normalisation is better. Replace TRANSFORM with TRANSFORM_BGR above.')
else:
    print('\n→ Standard normalisation is fine. No change needed.')

## Step 15 — Save results to JSON

In [ ]:
output = {
    'overall': {
        'mean_accuracy':  round(float(mean_acc),  4),
        'mean_f1':        round(float(mean_f1),   4),
        'mean_precision': round(float(mean_prec), 4),
        'mean_recall':    round(float(mean_rec),  4),
        'n_images':       len(eval_files),
        'threshold':      THRESHOLD,
    },
    'per_attribute': results,
}

with open('eval_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print('Saved: eval_results.json')
print(f'\nFinal summary:')
print(f'  Mean accuracy  : {mean_acc*100:.2f}%')
print(f'  Mean F1        : {mean_f1*100:.2f}%')